# Denoiser training: FullSubNet-like

Цель notebook:

- обучить отдельную модель подавления шума;
- использовать `FullSubNet-like` backbone, а не `TinyMaskNet`;
- сравнить `identity`, модель до обучения и модель после fine-tuning;
- послушать аудио до/после.

Важно: `base_init.pt` ниже — это **та же архитектура до обучения**, а не официальный pretrained FullSubNet checkpoint. Позже его можно заменить на внешний pretrained checkpoint, если подключим совместимую реализацию.


## 1. Настраиваем проект

Notebook рассчитан на VSCode Colab extension / Colab runtime. Если `/content/project` еще нет, первая ячейка сделает `git clone`.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess

REPO_URL = 'https://github.com/PushinMax/noise-suppression-mvp.git'


def run(command: str) -> None:
    print(f"\n$ {command}", flush=True)
    subprocess.run(command, shell=True, check=True, executable='/bin/bash')


def reset_paths(*paths: str) -> None:
    for raw_path in paths:
        path = Path(raw_path)
        if path.is_dir():
            shutil.rmtree(path)
            print(f'Удалена папка: {path}')
        elif path.exists():
            path.unlink()
            print(f'Удален файл: {path}')


def read_manifest(path: str) -> list[dict]:
    manifest_path = Path(path)
    if not manifest_path.exists():
        raise FileNotFoundError(f'Не найден manifest: {manifest_path}')
    return [json.loads(line) for line in manifest_path.read_text(encoding='utf-8').splitlines() if line.strip()]


def assert_nonempty_manifest(path: str) -> list[dict]:
    rows = read_manifest(path)
    if not rows:
        raise RuntimeError(f'Manifest пуст: {path}')
    print(f'{path}: rows = {len(rows)}')
    return rows


def is_project_root(path: Path) -> bool:
    return (path / 'pyproject.toml').exists() and (path / 'src/noise_suppression').exists()


def find_project_root(start: Path) -> Path | None:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if is_project_root(candidate):
            return candidate
    fallback = Path('/content/project')
    if is_project_root(fallback):
        return fallback
    return None


PROJECT_ROOT = find_project_root(Path.cwd())

if PROJECT_ROOT is None:
    if not REPO_URL:
        raise FileNotFoundError('Не найден проект. Укажите REPO_URL и перезапустите ячейку.')
    PROJECT_ROOT = Path('/content/project')
    if PROJECT_ROOT.exists():
        raise FileExistsError(f'{PROJECT_ROOT} уже существует, но это не корень проекта.')
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_ROOT)], check=True)

os.chdir(PROJECT_ROOT)
print('PROJECT_ROOT =', PROJECT_ROOT)
print('pyproject exists =', (PROJECT_ROOT / 'pyproject.toml').exists())
print('package exists =', (PROJECT_ROOT / 'src/noise_suppression').exists())


## 2. Устанавливаем зависимости

Для этого notebook нужны `train` и `data` extras. Метрики `SI-SDR` и `SNR` считаются без тяжелых дополнительных пакетов.


In [ ]:
run('pip -q install uv')
run('git pull')
run('uv sync --extra train --extra data --extra dev')
run('uv run noise-suppression env check')


## 3. Готовим данные

Размер выбран так, чтобы первый run помещался примерно в короткую Colab GPU-сессию:

- `160` clean Russian FLEURS clips;
- synthetic noise pool: fan / keyboard / traffic / cafe-babble-like;
- fallback включен только чтобы не блокироваться на временной проблеме Hugging Face.


In [ ]:
reset_paths(
    'data/denoiser_fullsubnet_seed',
    'manifests/denoiser_fullsubnet_clean.jsonl',
    'manifests/denoiser_fullsubnet_noise.jsonl',
)
Path('manifests').mkdir(exist_ok=True)

run('uv run python scripts/prepare_first_colab_dataset.py --output-root data/denoiser_fullsubnet_seed --num-clean 160 --seed 42 --allow-synthetic-fallback')
run('uv run noise-suppression manifest build data/denoiser_fullsubnet_seed/clean manifests/denoiser_fullsubnet_clean.jsonl --kind clean --speaker-depth 0')
run('uv run noise-suppression manifest build data/denoiser_fullsubnet_seed/noise manifests/denoiser_fullsubnet_noise.jsonl --kind noise')
assert_nonempty_manifest('manifests/denoiser_fullsubnet_clean.jsonl')
assert_nonempty_manifest('manifests/denoiser_fullsubnet_noise.jsonl')
run('uv run noise-suppression manifest summarize manifests/denoiser_fullsubnet_clean.jsonl')
run('uv run noise-suppression manifest summarize manifests/denoiser_fullsubnet_noise.jsonl')


## 4. Создаем synthetic noisy-clean пары

Конфиг:

- `2500` mixtures;
- `75% train / 25% val`;
- сегменты `2.5-4.0s`;
- SNR чаще попадает в диапазон `0-10 dB`.


In [ ]:
mix_config = {
    'seed': 42,
    'mixing': {
        'clean_manifest': '../manifests/denoiser_fullsubnet_clean.jsonl',
        'noise_manifest': '../manifests/denoiser_fullsubnet_noise.jsonl',
        'rir_manifest': None,
        'sample_rate': 16000,
        'num_examples': 2500,
        'min_duration_sec': 2.5,
        'max_duration_sec': 4.0,
        'snr_min_db': -2.0,
        'snr_max_db': 18.0,
        'focus_snr_min_db': 0.0,
        'focus_snr_max_db': 10.0,
        'focus_probability': 0.75,
        'reverb_probability': 0.0,
        'target_peak': 0.95,
    },
}

mix_config_path = Path('configs/denoiser_fullsubnet.runtime.yaml')
mix_config_path.write_text(json.dumps(mix_config, ensure_ascii=False, indent=2), encoding='utf-8')
print(mix_config_path)
print(mix_config_path.read_text(encoding='utf-8'))


In [ ]:
reset_paths(
    'manifests/denoiser_fullsubnet_mix_plan.jsonl',
    'data/denoiser_fullsubnet_rendered',
    'manifests/denoiser_fullsubnet_train.jsonl',
    'manifests/denoiser_fullsubnet_val.jsonl',
    'outputs/denoiser_fullsubnet_identity_val',
    'outputs/denoiser_fullsubnet_metrics',
)
Path('outputs/denoiser_fullsubnet_metrics').mkdir(parents=True, exist_ok=True)

run('uv run noise-suppression mix plan configs/denoiser_fullsubnet.runtime.yaml manifests/denoiser_fullsubnet_mix_plan.jsonl')
assert_nonempty_manifest('manifests/denoiser_fullsubnet_mix_plan.jsonl')
run('uv run noise-suppression mix render manifests/denoiser_fullsubnet_mix_plan.jsonl data/denoiser_fullsubnet_rendered --overwrite')
assert_nonempty_manifest('data/denoiser_fullsubnet_rendered/rendered_denoiser_fullsubnet_mix_plan.jsonl')
run('uv run noise-suppression manifest split data/denoiser_fullsubnet_rendered/rendered_denoiser_fullsubnet_mix_plan.jsonl manifests/denoiser_fullsubnet_train.jsonl manifests/denoiser_fullsubnet_val.jsonl --val-ratio 0.25 --seed 42')
assert_nonempty_manifest('manifests/denoiser_fullsubnet_train.jsonl')
assert_nonempty_manifest('manifests/denoiser_fullsubnet_val.jsonl')

run('uv run noise-suppression baseline apply manifests/denoiser_fullsubnet_val.jsonl outputs/denoiser_fullsubnet_identity_val --mode identity')
run('uv run noise-suppression metrics enhancement manifests/denoiser_fullsubnet_val.jsonl outputs/denoiser_fullsubnet_identity_val --output-path outputs/denoiser_fullsubnet_metrics/identity.json')


## 5. Выбираем метрики

В этом notebook считаем:

- `SI-SDR`: основная intrusive signal-level метрика для synthetic clean/noisy validation;
- `SNR`: простая waveform reconstruction метрика.

Позже добавим `WER` русского ASR и `DNSMOS/STOI` как более прикладные guardrails.


## 6. Конфиг модели FullSubNet-like

Это облегченная FullSubNet-like схема:

- full-band GRU branch;
- sub-band GRU branch;
- fusion через full-band cue + локальный frequency context;
- выход: spectral mask;
- восстановление: complex spectrogram * mask -> iSTFT.


In [ ]:
train_config = {
    'seed': 42,
    'data': {
        'train_manifest': '../manifests/denoiser_fullsubnet_train.jsonl',
        'val_manifest': '../manifests/denoiser_fullsubnet_val.jsonl',
        'sample_rate': 16000,
        'segment_seconds': 2.0,
        'batch_size': 16,
        'num_workers': 2,
        'limit_train': None,
        'limit_val': None,
    },
    'model': {
        'architecture': 'fullsubnet_lite',
        'n_fft': 512,
        'hop_length': 128,
        'win_length': 512,
        'full_hidden_size': 48,
        'sub_hidden_size': 24,
        'subband_context': 2,
        'full_num_layers': 1,
        'sub_num_layers': 1,
    },
    'training': {
        'epochs': 8,
        'learning_rate': 0.0003,
        'waveform_loss_weight': 1.0,
        'magnitude_loss_weight': 0.5,
        'device': 'auto',
        'output_dir': '../outputs/denoiser_fullsubnet_tuned',
        'checkpoint_mirror_dir': None,
        'save_every_epoch': True,
        'log_interval': 20,
    },
}

train_config_path = Path('configs/denoiser_fullsubnet_train.runtime.yaml')
train_config_path.write_text(json.dumps(train_config, ensure_ascii=False, indent=2), encoding='utf-8')
print(train_config_path)
print(train_config_path.read_text(encoding='utf-8'))


## 7. Сохраняем base checkpoint до обучения

Это точка сравнения “основа до fine-tuning”. Она показывает, что дает сама архитектура без обучения.


In [ ]:
reset_paths('outputs/denoiser_fullsubnet_base', 'outputs/denoiser_fullsubnet_base_val')
run("""uv run python - <<'PY'
from pathlib import Path
import torch
from noise_suppression.training import (
    build_model,
    checkpoint_model_config,
    resolve_experiment_config,
    set_seed,
)

config = resolve_experiment_config(Path('configs/denoiser_fullsubnet_train.runtime.yaml'))
set_seed(config.seed)
model = build_model(config)
output_dir = Path('outputs/denoiser_fullsubnet_base')
output_dir.mkdir(parents=True, exist_ok=True)
checkpoint = {
    'epoch': 0,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': {},
    'metrics': {'epoch': 0, 'note': 'FullSubNetLite before fine-tuning'},
    'model_config': checkpoint_model_config(model),
}
torch.save(checkpoint, output_dir / 'base_init.pt')
print(output_dir / 'base_init.pt')
print('parameters =', sum(p.numel() for p in model.parameters()))
PY""")

run('uv run noise-suppression train infer outputs/denoiser_fullsubnet_base/base_init.pt manifests/denoiser_fullsubnet_val.jsonl outputs/denoiser_fullsubnet_base_val')
run('uv run noise-suppression metrics enhancement manifests/denoiser_fullsubnet_val.jsonl outputs/denoiser_fullsubnet_base_val --output-path outputs/denoiser_fullsubnet_metrics/base_init.json')


## 8. Fine-tuning FullSubNet-like

Checkpoint сохраняется после каждой эпохи в `outputs/denoiser_fullsubnet_tuned`.


In [ ]:
reset_paths('outputs/denoiser_fullsubnet_tuned')
run('uv run noise-suppression train fit configs/denoiser_fullsubnet_train.runtime.yaml')


## 9. История обучения


In [ ]:
import matplotlib.pyplot as plt

history_path = Path('outputs/denoiser_fullsubnet_tuned/history.json')
history = json.loads(history_path.read_text(encoding='utf-8'))['history']

print('Checkpoints:')
for path in sorted(Path('outputs/denoiser_fullsubnet_tuned').glob('*')):
    print('  ', path.name)

required = ['best.pt', 'last.pt', 'history.json', 'resolved_config.json']
missing = [name for name in required if not (Path('outputs/denoiser_fullsubnet_tuned') / name).exists()]
assert not missing, f'Missing files: {missing}'

epochs = [row['epoch'] for row in history]
train_loss = [row['train_loss'] for row in history]
val_loss = [row['val_loss'] for row in history]
val_sisdr = [row['val_si_sdr'] for row in history]

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs, train_loss, marker='o', label='train_loss')
plt.plot(epochs, val_loss, marker='o', label='val_loss')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.grid(True)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, val_sisdr, marker='o', color='tab:green', label='val_si_sdr')
plt.xlabel('epoch')
plt.ylabel('SI-SDR')
plt.grid(True)
plt.legend()
plt.show()

history


## 10. Сравниваем base и fine-tuned модель


In [ ]:
reset_paths('outputs/denoiser_fullsubnet_tuned_val')
run('uv run noise-suppression train infer outputs/denoiser_fullsubnet_tuned/best.pt manifests/denoiser_fullsubnet_val.jsonl outputs/denoiser_fullsubnet_tuned_val')
run('uv run noise-suppression metrics enhancement manifests/denoiser_fullsubnet_val.jsonl outputs/denoiser_fullsubnet_tuned_val --output-path outputs/denoiser_fullsubnet_metrics/fine_tuned.json')

import pandas as pd

metrics_dir = Path('outputs/denoiser_fullsubnet_metrics')
rows = []
for name in ['identity', 'base_init', 'fine_tuned']:
    payload = json.loads((metrics_dir / f'{name}.json').read_text(encoding='utf-8'))
    rows.append({'model': name, **payload})
metrics_df = pd.DataFrame(rows)
metrics_df


## 11. Слушаем аудио

Сравниваем:

1. noisy input;
2. base FullSubNet-like до обучения;
3. fine-tuned FullSubNet-like;
4. clean target.


In [ ]:
from IPython.display import Audio, Markdown, display

val_rows = read_manifest('manifests/denoiser_fullsubnet_val.jsonl')
sample = val_rows[0]
base_path = Path('outputs/denoiser_fullsubnet_base_val') / f"{sample['id']}.wav"
tuned_path = Path('outputs/denoiser_fullsubnet_tuned_val') / f"{sample['id']}.wav"

print('Sample id:', sample['id'])
print('Noisy:', sample['noisy_path'])
print('Base init:', base_path)
print('Fine tuned:', tuned_path)
print('Clean:', sample['clean_path'])

for title, audio_path in [
    ('1. Noisy input', sample['noisy_path']),
    ('2. Base FullSubNet-like before training', str(base_path)),
    ('3. Fine-tuned FullSubNet-like', str(tuned_path)),
    ('4. Clean target', sample['clean_path']),
]:
    display(Markdown(f'**{title}**'))
    display(Audio(audio_path))


## 12. Что считать успехом первого запуска

Минимальный успех:

- `fine_tuned` лучше `base_init` по `SI-SDR` и `SNR`;
- желательно `fine_tuned` лучше `identity` по `SI-SDR`;
- на слух меньше шума без сильной деградации речи.

Если `fine_tuned` хуже `identity`, это не провал проекта: значит надо менять loss, noise mix, размер модели или количество данных.
